# 06 評価・再現性・最終レポート — 強化学習の結果を、どこまで信じてよいか

> 🔰 **やさしい導入**（この回のひとこと）
>
> - **ひとことで言うと**: 結果をどこまで信じてよいか。シードを変えたときのばらつきと、最終評価を確認する回です。
> - **たとえ話**: 同じ実験を何度も繰り返して、「たまたま」でないかを確かめる。
> - **先に読むと分かりやすい**: [00_basics.ipynb](00_basics.ipynb) の「2-7（ばらつきと複数回）・5」。用語が分からなくなったら、同じNotebookの「用語集」へ。
> - **難しければ**: 各コードの後ろにある「**出力の見方**」の文章だけ読めば、結論は追えます。

> ⏱ 所要時間: 説明を読む 15分 ＋ 実行 1分（最終評価の結果は、保存済みのものを読みます）

## ゴール
- 強化学習の評価で**守るべき約束**（複数シード・学習と評価の区別・ハイパーパラメータの選び方・最終評価は1回だけ）を、実際の結果で確認する。
- **最終評価（test）**の結果を読み、**Phase4 の結論を見直す**（強い言い方をしてはいけない理由）。
- **再現性**（同じシードなら同じ結果）と、**ばらつき**（シードを変えると変わる）を確かめる。

## 前のNotebookとのつながり
00〜05 で、迷路・バンディット・CartPole・蓄電池と、手法を一つずつ試しました。
ここでは、**設定を開発用のシードで選び、固定してから、未使用のシードで1回だけ評価**します。

## 評価の約束（このプロジェクトで守ったこと）
1. **学習中の成績**（探索あり）と、**学習後の方策の成績**（探索なし・別のシード）を区別する。
2. **複数シード**（10）で、平均とばらつき（sd）を出す。**差がばらつきより小さければ、差とは言わない**。
3. **ハイパーパラメータは、開発用のシード**（2M番台）で選ぶ。評価は**別のシード**。
4. **最終評価（3M番台）は、1回だけ**。結果を見て、設定を変えない（`TEST_USED.json` で、2回目を拒否する）。
5. 「**後から選んだ最高値**」（評価の最大）ではなく、**最後のネットワーク／方策**で評価する。
6. **基準**（ランダム・ルール・最適）と比べる。

## キーワード
- **開発用シード／最終評価用シード**: 設定を選ぶ用／最後に1回だけ成績をつける用。重ならない。
- **正規化した成績**: 0＝ランダム、1＝最適（動的計画法）。課題をまたいで比べられる。
- **標準誤差**: 平均の誤差の目安（sd ÷ √シード数）。差がこの2〜3倍より小さいと、はっきりした差とは言えない。

In [ ]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
FIN = ROOT / "outputs" / "final"
need = [FIN / "dev_selection.json", FIN / "final_val.csv", FIN / "final_test.csv", FIN / "TEST_USED.json"]
missing = [f.name for f in need if not f.exists()]
if missing:
    raise SystemExit(f"保存済みの結果がありません: {missing}。ターミナルで src/select_dev.py と src/final_eval.py（--split val、--split test）を実行してください（約10分）。")
dev = json.loads((FIN / "dev_selection.json").read_text(encoding="utf-8"))
used = json.loads((FIN / "TEST_USED.json").read_text(encoding="utf-8"))
print("最終評価（test）の使用日時:", used["used_at"])
print("固定した設定:", json.dumps(used["config"], ensure_ascii=False))

**何をしているか**: 最終評価の記録（`TEST_USED.json`）を読みます。**固定した設定**が、記録されています。

**出力の見方**: 最終評価で使った設定（学習率・DQNの設定など）が、**評価の前に固定された**ことが、記録として残っています。このファイルがある間、`final_eval.py --split test` は、**拒否**されます。

In [ ]:
rows = []
for name, v in dev["dqn_table"].items():
    rows.append({"候補": name, "最後の3回の平均（開発用シード5つ）": round(v, 1), "選んだ": "◎" if name == dev["dqn_name"] else ""})
print("開発用シードで選んだ DQN の設定:")
display(pd.DataFrame(rows).set_index("候補"))
print("表形式の学習率（正規化した成績。1＝最適）:")
display(pd.Series(dev["tabular_table"], name="正規化した成績").round(3).to_frame())
print("蓄電池のQ学習の学習率（電気代 円/日。小さいほど良い）:", {k: round(v, 1) for k, v in dev["battery_table"].items()}, "→ 選択 α =", dev["battery_alpha"])

**何をしているか**: 開発用のシード（2M番台、5つ）で、最終評価に使う設定を**選んだ結果**を見ます。

**出力の見方**:
- **DQN**: 「**安定重視**（学習率3e-4・ターゲット更新1000）」が**182.9**で最良。「既定」は175.8。「**箱を大きく（5万）**」は**118.7**、「安定重視＋箱5万＋600回」は103.5と、**悪い**。
- 注意: Phase4 の予備実験（別の6シード）では、「箱を大きく（5万）」が**約170**と、**良く見えました**。新しい開発用シードでは**悪く**なっています。つまり、**ハイパーパラメータの優劣は、シードが変わるとひっくり返るほどぶれる**のです。「効いた」と思った工夫が、**運だった**ことがあります。
- **表形式**: 学習率α=0.05〜0.1 が最適（正規化1.000）。α=0.5 は少し悪い（SARSAで0.986）。Phase2 で、すべる崖が解けなかったのは、**学習率が大きすぎた**ためでした。
- **蓄電池のQ学習**: α=0.2（252.6円）が、α=0.05（282.1円）より良い。学習が遅い（α小）と、3000エピソードでは足りない。

In [ ]:
val = pd.read_csv(FIN / "final_val.csv"); test = pd.read_csv(FIN / "final_test.csv")
cols = ["課題", "手法", "平均", "sd", "正規化", "失敗シード", "満点に近いシード", "節約の割合"]
display(test[cols].round(3).set_index(["課題", "手法"]).dropna(how="all"))

## 最終評価（test）の結果

**何をしているか**: 固定した設定で、**未使用のシード**（学習3.0M番台の10シード、評価3.1M）で、**1回だけ**評価した結果です。

**出力の見方**:
- **迷路・崖（表形式）**: 正規化した成績は、**Q学習もSARSAも、ほぼ1.000（最適）**。maze は −12、cliff は Q学習が −13（最適）・SARSA が −15（崖から離れた安全な道）、**すべる崖でも Q学習 −30.25、SARSA −30.98**（最適 −29.77）。学習率を下げて（0.05）、探索を減らし、エピソードを増やす（2000）と、**Phase2 で届かなかった課題が解けます**。失敗したシードは0。
- **CartPole**: ルールは満点200、ランダムは22。**離散化した表形式 158.9（±44.2）**、**DQN 172.1（±27.3）**。どちらも、**10シード中、満点に近い（195以上）のは3つだけ**。
- **蓄電池**: 何もしない 351.9円/日、**ルール 232.8**（節約の91.2%）、**動的計画法（最適）221.3**、**Q学習 234.6（±8.2）**（節約の約90%）。

In [ ]:
m = val.merge(test, on=["課題", "手法"], suffixes=("_val", "_test"))
m = m[m["sd_test"].notna()][["課題", "手法", "平均_val", "平均_test", "sd_val", "sd_test"]].round(2)
m

**何をしているか**: 予行演習（val。学習2.2M・評価2.3M）と、最終評価（test）の、**平均とばらつき**を並べます。

**出力の見方**:
- 表形式・蓄電池は、**valとtestで、ほぼ同じ**（正規化1.000、蓄電池のQ学習 243.6 と 234.6。基準が変わる＝天気の並びが違うため、絶対値は違う）。
- **CartPole は、大きく動きます**。離散化: **187.4（val）→ 158.9（test）**、DQN: **168.3（val）→ 172.1（test）**。**シードの組が変わるだけで、平均が約30動く**のです（sdが27〜44）。
- つまり、CartPole の結果は、**10シードでも、まだ、ぶれが大きい**。順位の議論は、慎重にすべきです。

In [ ]:
def compare(task, a, b, n=10):
    ra = test[(test["課題"] == task) & (test["手法"] == a)].iloc[0]
    rb = test[(test["課題"] == task) & (test["手法"] == b)].iloc[0]
    sea, seb = ra["sd"] / np.sqrt(n), (rb["sd"] / np.sqrt(n) if not np.isnan(rb["sd"]) else 0.0)
    diff = ra["平均"] - rb["平均"]
    se = float(np.sqrt(sea ** 2 + seb ** 2))
    return {"課題": task, "比較": f"{a} − {b}", "平均の差": round(float(diff), 1), "差の標準誤差": round(se, 1),
            "差÷標準誤差": round(float(diff / se), 1), "判定": "区別できない" if abs(diff / se) < 2 else "差がある"}

pd.DataFrame([
    compare("CartPole", "DQN（最後のネットワーク）", "離散化した表形式Q学習"),
    compare("蓄電池（円/日）", "Q学習", "ルール"),
]).set_index("課題")

## 差は、本当にあるのか

**何をしているか**: 「DQN と離散化」「Q学習 とルール」の差を、**標準誤差**（sd ÷ √10）と比べます。差が標準誤差の**2倍より小さければ、区別できません**。

**出力の見方**:
- **CartPole: DQN − 離散化 = 約+13**（DQNが上）。標準誤差は約16で、**差÷標準誤差は 約0.8**。**区別できません**。
- Phase4 で、私は「離散化した表形式のほうが、DQNより安定して強かった」と書きました。**最終評価では、順位が逆**です。正しくは、「**この問題で、DQNと離散化は、同程度**。**ルール（満点）には、どちらも及ばない**」。**1回の実験の順位を、一般的な結論にしてはいけません**。
- **蓄電池: Q学習 − ルール = 約+1.8円/日**（Q学習のほうが高い＝やや悪い）。標準誤差は約2.6で、**区別できません**。「Q学習は、ルールに並ぶ」が、正確な言い方です（ルールを超えるとも、下回るとも言えない）。
- **評価で使った天気の並び**は、すべての方策で同じ（対応のある比較）なので、ルールと最適方策の差（11円/日）は、**天気のせいではなく、方策の差**です。

In [ ]:
from tabular import train
from rl_envs import make_env, evaluate_policy
from tabular import greedy_policy
import final_eval

def one(seed):
    env = make_env("cliff", slip=0.2, seed=seed)
    r = train(env, "q", episodes=2000, alpha=0.05, gamma=1.0, seed=seed, **final_eval.FINAL_CONFIG["tabular_eps"])
    return evaluate_policy(env, greedy_policy(r["Q"]), 100, seed=3_100_000)["mean_return"]

print("同じシード（3000000）で2回:", round(one(3_000_000), 4), round(one(3_000_000), 4), "→ 同じ")
print("シードを変える:", [round(one(3_000_000 + i), 2) for i in range(5)], "→ ばらつく")

## 再現性とばらつき

**何をしているか**: すべる崖で、Q学習を、(1) **同じシードで2回**、(2) **シードを変えて5回**、学習・評価します。

**出力の見方**:
- **同じシードなら、完全に同じ結果**（小数点以下まで一致）。**乱数のシードを固定している**ので、**再現できます**。
- **シードを変えると、結果が変わります**。5つのうち、**3つは同じ値（−29.34、ほぼ最適）に落ち着き、2つは −33 台**の、少し劣る方策になりました。強化学習は、**乱数（探索・環境・初期値）でぶれる**ので、**1つのシードの結果は、運を含みます**。だから、**複数シードの平均とばらつき**で見るのです。
- 再現性は、**同じ条件の再実行ができる**ことを保証します。しかし、**結論が一般的**かは、別の問題です（別のシード・別の設定でも、同じ結論になるか）。

## 試してみよう（演習）
1. **シードの数を増やす**: `final_eval.py` の `SPLITS["val"]["train_seeds"]` を、10個から30個にして（`range(30)`）、`--split val` を実行してください。CartPoleの平均とsdが、どう変わるかを見ます。分かるのは、シードを増やすと、平均が安定し、標準誤差が小さくなることです（順位が、はっきりするか見ましょう）。
2. **後から選んだ最高値を使ってみる**: `run_phase4.py` の評価で、最後の3回の平均の代わりに「最大」を使った表と、比べてください。分かるのは、最高値は、実力より**ずっと良く見える**ことです（Phase4 の「最大」の列は、いずれも200近い）。
3. **開発用シードで、別の設定を試す**: `select_dev.py` の `DQN_CANDIDATES` に、自分の設定（隠れ層の大きさ `hidden`、探索率の減り方など）を足して、選ばれるか見てください。ただし、**最終評価（test）は、もう使えません**（再評価は、4M番台などの新しいシードで）。
4. **ルールと比べる**: CartPole で、DQN がルール（200）に届かない理由を、考えてみましょう。学習のエピソード数、探索率、報酬の設計（棒の角度に応じた報酬など）のどれが効きそうですか。

## まとめ
- **最終評価**: 表形式（迷路・崖）は、Q学習・SARSAともに**最適**。CartPole は、DQN **172**・離散化 **159**（ルール200、ランダム22）で、**区別できない**。蓄電池は、Q学習 **234.6円/日**、ルール **232.8**、最適 **221.3**（何もしない351.9）で、Q学習とルールは**区別できない**。
- **Phase4 の結論は、見直す**: 「離散化がDQNより強い」は、**1回の実験の順位**で、最終評価では**逆転**した。**差は誤差の範囲**。
- **ハイパーパラメータの優劣は、シードでぶれる**（「箱を大きく」は、別のシードで良く見えた→悪かった）。**開発用シードと評価用シードを分ける**意味がある。
- **後から選んだ最高値**は、実力ではない。**最後の方策**で評価する。
- **再現性**（同じシードで同じ結果）は、**固定したシード**で実現できる。ただし、**結論の一般性**は、別の問題（複数シード・基準との比較で確かめる）。
- 強化学習は、**ぶれやすく、設定に敏感**。結果は、**複数シードの平均とばらつき**、**基準との比**で、慎重に読む。

最終レポートは [REPORT.md](../REPORT.md) にあります。